# Batman Model Verification

For each exported transit, plots:
- **Top**: raw flux (dots) + batman model (line) — checks alignment and depth
- **Bottom**: residuals (raw − model inside transit, flux−1 outside) — should be noise around zero with no bowl/bump

Red shading = transit window (±T14/2 from tc). If the model is good, residuals inside the red zone should look the same as residuals outside.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import batman
import os, glob, warnings
warnings.filterwarnings('ignore')

# ── Parameters (edit for each planet) ────────────────────────────────────────
PLANET   = 'Qatar-1'
SECTOR   = 24
PERIOD   = 1.42002504
K        = 1.143 * 0.10045 / 0.803   # Rp/Rs
A_RS     = 6.245
INC      = 84.0
ECC      = 0.0
U1, U2   = 0.44, 0.25
T14_HR   = 2.00
T_DAYS   = T14_HR / 24.0

TRANSIT_ROOT = f'../transit_csvs/{PLANET}/sector_{SECTOR:03d}'
transit_dirs = sorted(glob.glob(f'{TRANSIT_ROOT}/transit_*/'))
print(f'Found {len(transit_dirs)} transits for {PLANET} S{SECTOR}')
print(f'k={K:.4f}  a/Rs={A_RS}  inc={INC}°  T14={T14_HR}hr')

In [ ]:
def batman_flux(tarr, tc):
    p = batman.TransitParams()
    p.t0 = tc; p.per = PERIOD; p.rp = K
    p.a  = A_RS; p.inc = INC; p.ecc = ECC; p.w = 90.0
    p.u  = [U1, U2]; p.limb_dark = 'quadratic'
    try:
        return batman.TransitModel(p, tarr).light_curve(p)
    except Exception:
        return np.ones_like(tarr)

print('Batman helper ready.')

In [ ]:
NCOLS = 4
nrows = int(np.ceil(len(transit_dirs) / NCOLS))
fig_top, axes_top = plt.subplots(nrows, NCOLS, figsize=(5*NCOLS, 3.5*nrows), squeeze=False)
fig_res, axes_res = plt.subplots(nrows, NCOLS, figsize=(5*NCOLS, 3.5*nrows), squeeze=False)

fig_top.suptitle(f'{PLANET} S{SECTOR} — Raw flux + Batman model', fontsize=13, fontweight='bold')
fig_res.suptitle(f'{PLANET} S{SECTOR} — Residuals (should be flat noise, no bowl/bump in red zone)', fontsize=13, fontweight='bold')

diag_rows = []  # for summary table

for idx, tdir in enumerate(transit_dirs):
    transit_label = os.path.basename(tdir.rstrip('/'))
    ax_t = axes_top[idx // NCOLS][idx % NCOLS]
    ax_r = axes_res[idx // NCOLS][idx % NCOLS]

    try:
        merged = pd.read_csv(os.path.join(tdir, 'merged_residuals.csv'))
        raw    = pd.read_csv(os.path.join(tdir, 'in_transit_raw.csv'))
        pre    = pd.read_csv(os.path.join(tdir, 'pre_baseline.csv'))
        post   = pd.read_csv(os.path.join(tdir, 'post_baseline.csv'))
        model_df = pd.read_csv(os.path.join(tdir, 'in_transit_model.csv'))
    except Exception as e:
        ax_t.text(0.5, 0.5, f'Load error:\n{e}', transform=ax_t.transAxes,
                  ha='center', va='center', fontsize=8)
        continue

    t_in  = raw['time_btjd'].values
    f_in  = raw['flux'].values
    t_mod = model_df['time_btjd'].values
    f_mod = model_df['flux_model'].values

    # tc = time of model minimum
    tc = t_mod[np.argmin(f_mod)]
    half_T = T_DAYS / 2

    # Recompute model on dense grid for smooth line
    t_all  = merged['time_btjd'].values
    f_all  = np.concatenate([pre['flux'].values,
                              f_in,
                              post['flux'].values])
    t_dense = np.linspace(t_all.min(), t_all.max(), 500)
    m_dense = batman_flux(t_dense, tc)
    # Rescale model to data baseline
    out_mask = np.abs(t_dense - tc) > 0.6*T_DAYS
    if out_mask.sum() > 3:
        baseline_data  = np.median(np.concatenate([pre['flux'].values, post['flux'].values]))
        baseline_model = np.median(m_dense[out_mask])
        m_dense = m_dense * (baseline_data / baseline_model)

    # ── Top panel: raw + model ────────────────────────────────────────────────
    ax_t.axvspan(tc - half_T, tc + half_T, alpha=0.10, color='red')
    ax_t.plot(pre['time_btjd'], pre['flux'], '.', color='#aaa', ms=2, alpha=0.6)
    ax_t.plot(post['time_btjd'], post['flux'], '.', color='#aaa', ms=2, alpha=0.6)
    ax_t.plot(t_in, f_in, '.', color='#c0392b', ms=2.5, alpha=0.8, label='raw')
    ax_t.plot(t_dense, m_dense, '-', color='blue', lw=1.5, alpha=0.9, label='batman')
    ax_t.axvline(tc, color='blue', lw=0.8, ls='--', alpha=0.7)
    ax_t.set_title(f'{transit_label}  tc={tc:.4f}', fontsize=8)
    ax_t.tick_params(labelsize=7)
    ax_t.legend(fontsize=6, loc='lower right')

    # ── Bottom panel: residuals ───────────────────────────────────────────────
    resid_pre  = merged[merged['segment']=='pre_baseline']
    resid_in   = merged[merged['segment']=='in_transit']
    resid_post = merged[merged['segment']=='post_baseline']

    ax_r.axhline(0, color='gray', lw=0.8, ls='--')
    ax_r.axvspan(tc - half_T, tc + half_T, alpha=0.10, color='red')
    ax_r.plot(resid_pre['time_btjd'],  resid_pre['residual'],  '.', color='#aaa', ms=2, alpha=0.6)
    ax_r.plot(resid_post['time_btjd'], resid_post['residual'], '.', color='#aaa', ms=2, alpha=0.6)
    ax_r.plot(resid_in['time_btjd'],   resid_in['residual'],   '.', color='#c0392b', ms=2.5, alpha=0.8)
    ax_r.set_title(f'{transit_label}', fontsize=8)
    ax_r.tick_params(labelsize=7)

    # ── Diagnostics ───────────────────────────────────────────────────────────
    resid_in_vals   = resid_in['residual'].values
    resid_base_vals = np.concatenate([resid_pre['residual'].values,
                                      resid_post['residual'].values])
    mean_in   = np.mean(resid_in_vals)
    std_in    = np.std(resid_in_vals)
    std_base  = np.std(resid_base_vals)
    # Flag if mean residual inside transit is > 1 sigma of baseline — possible misalignment
    flag = '⚠' if abs(mean_in) > std_base else 'OK'
    diag_rows.append({
        'transit': transit_label,
        'tc': round(tc, 5),
        'mean_resid_in': round(mean_in, 6),
        'std_resid_in':  round(std_in, 6),
        'std_baseline':  round(std_base, 6),
        'status': flag
    })

# Hide unused panels
for i in range(len(transit_dirs), nrows*NCOLS):
    axes_top[i//NCOLS][i%NCOLS].set_visible(False)
    axes_res[i//NCOLS][i%NCOLS].set_visible(False)

fig_top.tight_layout(rect=[0,0,1,0.96])
fig_res.tight_layout(rect=[0,0,1,0.96])

os.makedirs('../results', exist_ok=True)
fig_top.savefig(f'../results/batman_verify_{PLANET}_S{SECTOR}_overlay.png', dpi=120, bbox_inches='tight')
fig_res.savefig(f'../results/batman_verify_{PLANET}_S{SECTOR}_residuals.png', dpi=120, bbox_inches='tight')
plt.show()

# ── Summary table ─────────────────────────────────────────────────────────────
diag_df = pd.DataFrame(diag_rows)
print('\n=== Alignment diagnostics ===')
print('mean_resid_in: mean residual inside transit window (should be ~0)')
print('std_baseline:  noise level of out-of-transit residuals')
print('status: OK if |mean_resid_in| < std_baseline, ⚠ if possible misalignment')
print()
print(diag_df.to_string(index=False))